# CMPDI / Coal India Limited: Domain-Specific Fine-Tuning of Qwen 2.5 3B with LoRA
### Smart India Hackathon (SIH) — Automated Extraction & Reporting Platform for Ministry of Coal

This notebook fine-tunes **Qwen 2.5 3B Instruct** on domain-specific coal mining reports (CMPDI borehole logs, NLC mine equipment stoppage reports, CIL quarterly production statements, and Parliamentary Q&As).

#### Key Architecture:
- **Base Model:** `Qwen/Qwen2.5-3B-Instruct`
- **Quantization:** 4-bit NormalFloat4 (NF4) with double quantization via `bitsandbytes`
- **Adaptation:** Low-Rank Adaptation (LoRA) with rank r=16, alpha=32, targeting linear projection layers (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`)
- **Trainable Parameters:** ~18M (less than 0.6% of base model weights)
- **Execution Time:** ~3-4 minutes on a free Google Colab T4 GPU

## Step 1: Install Required Libraries & Check GPU

In [ ]:
!pip install -q --upgrade pip
!pip install -q torch transformers peft datasets trl bitsandbytes accelerate

import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Warning: Running on CPU. Please switch Runtime to GPU: Runtime -> Change runtime type -> T4 GPU")

## Step 2: Load and Verify Cleaned Training Dataset (train.jsonl)

In [ ]:
import json
import os
from datasets import Dataset

# Check if train.jsonl was uploaded
if not os.path.exists("train.jsonl"):
    print("Please upload your train.jsonl file into the Colab file tree on the left sidebar.")
    try:
        from google.colab import files
        files.upload()
    except Exception:
        pass

with open("train.jsonl", "r", encoding="utf-8") as fh:
    raw_data = [json.loads(line) for line in fh]

print(f"[+] Loaded {len(raw_data)} high-confidence training pairs.")
print("Sample 1 text snippet:", raw_data[0]["text"][:200])
print("Sample 1 fields keys:", list(raw_data[0]["fields"].keys()))

## Step 3: Load Base Model with 4-bit Quantization (QLoRA)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model

model_id = "Qwen/Qwen2.5-3B-Instruct"
print(f"[*] Loading tokenizer: {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("[*] Configuring 4-bit NormalFloat4 quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 4: Format Training Data into ChatML Instruction Template

In [ ]:
formatted_records = []
system_prompt = (
    "You are CMPDI's expert geological and mining document extraction engine. "
    "Extract all verified production figures, borehole reserves, depths, and operational metrics into structured JSON."
)

for item in raw_data:
    text_content = item.get("text", "").strip()
    fields = item.get("fields", {})
    if not text_content or not fields:
        continue
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Extract structured figures from this document text:\n\n{text_content}"},
        {"role": "assistant", "content": json.dumps(fields, ensure_ascii=False, indent=2)},
    ]
    formatted_records.append({"text": tokenizer.apply_chat_template(messages, tokenize=False)})

train_dataset = Dataset.from_list(formatted_records)
print(f"[+] Total prepared training dataset size: {len(train_dataset)} examples")

## Step 5: Configure Hyperparameters & Run SFTTrainer

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=5,
    save_strategy="epoch",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args,
)

print("[*] Commencing QLoRA training...")
trainer.train()
print("[+] Training finished successfully!")

## Step 6: Save LoRA Adapter & Download for Deployment

In [ ]:
import shutil

adapter_dir = "cmpdi_qwen_adapter"
print(f"[*] Saving adapter weights to {adapter_dir}...")
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

# Create zip archive for 1-click download
shutil.make_archive("cmpdi_qwen_adapter", "zip", adapter_dir)
print("[+] Created cmpdi_qwen_adapter.zip!")

try:
    from google.colab import files
    files.download("cmpdi_qwen_adapter.zip")
    print("[+] Triggered download in browser!")
except Exception as e:
    print("Download manually from left sidebar: cmpdi_qwen_adapter.zip")

## Step 7: Benchmark Inference on Unseen Test Document

In [ ]:
test_doc = """NLC INDIA LTD
Stoppage Report for MINE-I
Date: 12.09.2026
NSB/BWE-1140 / OVER BURDEN / TWH: 18.00 / EWH: 12.50 / Output: 1420.500 / Rate: 113.640
Stoppage From Stoppage To Duration Stoppage Description
0600 0730 1.50 Shift change and pre-start inspection
1100 1345 2.75 Chute jam clearance at transfer point TP-4
1800 1915 1.25 Power supply tripping - Substation 3 feeder
"""

test_prompt = f"Extract structured figures from this document text:\n\n{test_doc}"
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_prompt},
]
input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=False)

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== EXTRACTED JSON OUTPUT FROM FINE-TUNED MODEL ===")
print(response)